# What are the following (write in text): True model error, in-sample error, out-of-sample error? Why can’t we know the true model error most of the time (in practice)?

1. True model error is a metric that calculates the error of a model tested on the entire possible inout it can have, so for example a model that classifies spam or not spam emails, if you somehow generated all the possible emails and their label it would classify them and then you would calculate the true error. So its based on the entire distribution of the input data possible.

2. In-sample error is the measure of the error of the model on the data it trained on so the data it has seen and that it tries to capture the function that with the input gives the coresponding output how many out of all the training samples does it get wrong.

3. Out-sample is the measure of the error of the model on data that is did not see before but is in the population of the same training data, something to consider is that both the training and the testing data are assumed (if large enough) to represent the entirety of the possible data so a representative sample.

So why is it almost impossible to get the true model error, because in almost every case you would need to generate impossible data so in the spam example you would need to get every email that has ever been written and that will ever be written which is infinte so we cannot gather or create the data in most cases to get the true model error, but with the assumption that the training and testing data are a respresentative sample it is ok to approximatly say that the test error (out-sample error) is almost equal to the true model error.

# What is the difference between a local vs a global optimum? Write in text.

When trying to get an optimal solution for some mathematical problem such as minimising a loss function of a model, the objective is the find the parameters that yield the lowest possible loss. To find this solution you need to find what is called a global optimum (in this case a minimum) the issue is when for example doing gradient descent algorithm the it can get stuck at a local minimum which is a point that appears to be the best solution it can find because moving in any direction yields a worse solution right now. So the if an algorithm finds a local optimum before finding the globa one it will return it as the optimal solution when in fact there is a better solution but it could not reach it.
So a global optimum (max or min) is the optimal solution with no better solution that can be found in your entire search space, a local optimum is a point that is better than every solution near it or in its neighbourhood and so to find the best solution it is easier when the function you are trying to find the optimal parameters for is convex which means it has one point the is optimal (higher or lower than all other points) but when the function is not convex you need to use other ways to try to find the global optimal like stochastic gradient descent for loss functions that are not convex ( stochastic gradient descent is a random gradient descent algorithm that starts from random points in the search space so that it can be done multiple times and hopefully can one time start near the global optimum)

# What is the hypothesis space? Write in text.

It is the entire possible space for the parameters of your function or in more general terms it is the entire valid space for the algorithm to search through and find the optimal solution, it has every possible solution in it that can satisfy the problem.

# What is bias-variance trade-off? Give an example where you have low bias and high variance and an example where you have low variance but high bias. Write in text.

What is bias?

Bias is the systematic error caused by a model making assumptions that are too restrictive to properly represent the underlying relationship in the data. High bias is associated with underfitting.

What is variance?

Variance measures how sensitive a model is to changes in its training data. A high-variance model can change significantly when trained on different samples from the same population and is associated with overfitting.

What is the bias-variance trade-off?

Increasing model complexity generally reduces bias because the model can represent more complicated relationships, but it can increase variance because the model becomes more sensitive to the training data. The goal is to find a balance where the model captures the real pattern while still generalizing well to unseen data.

An example of high bias, low variance is a model that is simple, simple enough to not be able to represent data that is curved (polynomial) but since it is simple it produces similar outputs on different training data.

An example of low bias, high variance is model that is a very large polynomial model that is able to capture very specific details about data so training error is low but changing the data even if not large difference could make the output dramatically different meaning it is capturing some error and thinking it is a pattern in the data.

In both the previous examples the idea is when changing the data it is sampled from the same population but the change would be in the part of the data that is reffered to as irreducable error, which cannot be removed from the data, but the general pattern in the data should be the same.

# What does regularization do? What is the difference between ridge, lasso, and elastic net? Write your answers in text.

It is a process that helps models generalize better, or achieve a better balance of variance and bias that makes the model perform better if done correctly, essentially it is a term added to the loss function (can be different according to type of regularization) that restricts the feature weights from becoming very large and that has the advantage of reducing the drastic change in output when a small difference is made in the input data. So there is Ridge, Lasso, and Elastic net (which is a combination of the two previous ways).

1. Ridge, uses what is called the L2 regularization which is a term added to the loss function that is the square sum of the weights (the goal is to minimise the loss function) so it makes sure they wouldn't grow alot. It looks like this Jridge​=J(w)+λsum(w^2) for all weights. The lambda controls how aggressive that reularization term is the higher it is the more aggressive and lower the values of the wieghts will be.

2. Lasso, uses what is called the L1 regularization which is also a term added to the loss function that is the sum of the absolute value of the weights so it also does not allow them to grow very large. It loos like this Jlasso=J(w)+λsum(abs(w)) for all weights. The difference between lasso and ridge is that ridge has only the ability to push weights very close to zero but not zero, but with lasso a weight that seems to have no impact when small keeps getting smaller and coud become zero meaning that entire feature has been removed so lasso has the ability to do feature selection.

3. Elastic net is a mix of both L1 and L2, the reason for this is while lasso seems better because it can basically do feature selection it is not always correct and can make the result worse in situations like having many features that are correlated, it tends to keep one of those features, so ridge can handle groups of correlated features better, so elastic net has both terms and has a hyper-paramter called l1_ratio which if it is set to 1 means it is entirely lasso and if it is set to 0 it means it is entirely ridge and so changing this ratio to finding a good balance is an issue for hyper-parameter tuning.


Regularization helps control overfitting by reducing a model’s effective complexity. A very complex model can have low bias but high variance, meaning it fits the training data very well but is too sensitive to that specific sample. Regularization discourages large coefficients, making the model less sensitive and usually reducing variance, although this can increase bias slightly. The goal is therefore to use enough regularization to improve performance on unseen data without making the model so simple that it underfits.

## Applying Ridge, Lasso, and Elastic Net Regularization

The following experiments reuse the Topic 8 IMDB classification setup. All four models use the same training data, test data, and TF-IDF features so that only the regularization method changes. The hyperparameters are fixed examples; no tuning or model-selection procedure is performed.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

# Locate and reuse the custom NLP components used in Topic 8.
current_path = Path.cwd().resolve()
repository_root = next(
    (path for path in (current_path, *current_path.parents)
     if (path / "phase-1-machine-learning-nlp").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

phase_1 = repository_root / "phase-1-machine-learning-nlp"
part_02 = phase_1 / "02-preprocessing-tokenization"
part_06 = phase_1 / "06-vectorization"
sys.path.insert(0, str(part_02 / "src"))
sys.path.insert(0, str(part_06 / "src"))

from preprocessor import preprocess_text
from tokenizer import regex_tokenize
from vectorizer import CustomTfidfVectorizer

# Reproduce the Topic 8 dataset preparation and split.
imdb = pd.read_csv(part_02 / "data" / "IMDB Dataset.csv")
imdb = imdb.dropna(subset=["review", "sentiment"])
imdb = imdb.drop_duplicates(subset="review").reset_index(drop=True)
imdb["label"] = imdb["sentiment"].map({"negative": 0, "positive": 1})

X_train, X_test, y_train, y_test = train_test_split(
    imdb["review"], imdb["label"], test_size=0.20,
    random_state=42, stratify=imdb["label"],
)

# Fit once on training reviews and reuse the same feature matrices for every model.
text_vectorizer = CustomTfidfVectorizer(
    preprocessor=preprocess_text, tokenizer=regex_tokenize,
    min_df=5, max_df=0.95, max_features=30_000,
)
X_train_tfidf = text_vectorizer.fit_transform(X_train)
X_test_tfidf = text_vectorizer.transform(X_test)

print(f"Training reviews: {len(X_train):,}")
print(f"Test reviews: {len(X_test):,}")
print(f"TF-IDF features: {X_train_tfidf.shape[1]:,}")


Training reviews: 39,665
Test reviews: 9,917
TF-IDF features: 30,000


In [2]:
# C=1.0 is used for all regularized models without hyperparameter tuning.
models = {
    # C=infinity removes the coefficient penalty for the comparison baseline.
    "Baseline Logistic Regression": LogisticRegression(
        C=np.inf, solver="lbfgs", max_iter=2_000, random_state=42
    ),
    # Ridge applies only the L2 penalty and generally shrinks all coefficients.
    "Ridge / L2": LogisticRegression(
        C=1.0, l1_ratio=0.0, solver="lbfgs",
        max_iter=2_000, random_state=42,
    ),
    # Lasso applies only the L1 penalty and can set coefficients to zero.
    "Lasso / L1": LogisticRegression(
        C=1.0, l1_ratio=1.0, solver="liblinear",
        max_iter=2_000, random_state=42,
    ),
    # Elastic Net mixes L1 and L2 equally and requires the SAGA solver.
    "Elastic Net": LogisticRegression(
        C=1.0, l1_ratio=0.5, solver="saga",
        max_iter=2_000, random_state=42,
    ),
}

metric_rows = {}
for model_name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    positive_probabilities = model.predict_proba(X_test_tfidf)[:, 1]

    # These are the same five test metrics reported in Topic 8.
    metric_rows[model_name] = {
        "accuracy": accuracy_score(y_test, predictions),
        "positive precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "positive recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "positive F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, positive_probabilities),
    }

metrics_comparison = pd.DataFrame.from_dict(metric_rows, orient="index")
metrics_comparison.index.name = "model"
display(metrics_comparison.style.format("{:.4f}"))


,accuracy,positive precision,positive recall,positive F1,ROC-AUC
model,,,,,
Baseline Logistic Regression,0.8925,0.8910,0.8953,0.8932,0.9568
Ridge / L2,0.8727,0.8585,0.8937,0.8758,0.9437
Lasso / L1,0.8639,0.8476,0.8885,0.8676,0.9358
Elastic Net,0.8665,0.8493,0.8923,0.8703,0.9391


In [3]:
# Measure coefficient sparsity for each regularized model.
coefficient_rows = []
for model_name in ["Ridge / L2", "Lasso / L1", "Elastic Net"]:
    coefficients = models[model_name].coef_.ravel()
    zero_count = np.count_nonzero(coefficients == 0.0)
    coefficient_rows.append({
        "model": model_name,
        "total coefficients": coefficients.size,
        "zero coefficients": zero_count,
        "zero coefficients (%)": 100 * zero_count / coefficients.size,
    })

coefficient_comparison = pd.DataFrame(coefficient_rows).set_index("model")
display(coefficient_comparison.style.format({
    "total coefficients": "{:,}",
    "zero coefficients": "{:,}",
    "zero coefficients (%)": "{:.2f}%",
}))


,total coefficients,zero coefficients,zero coefficients (%)
model,,,
Ridge / L2,"30,000",0,0.00%
Lasso / L1,"30,000","29,586",98.62%
Elastic Net,"30,000","27,938",93.13%


### Observations

- With the fixed hyperparameters used here, all five test metrics decreased for each regularized model compared with the unregularized baseline.
- Baseline accuracy was 0.8925. Ridge/L2 accuracy was 0.8727, Lasso/L1 accuracy was 0.8639, and Elastic Net accuracy was 0.8665.
- Positive-class F1 decreased from 0.8932 for the baseline to 0.8758 for Ridge/L2, 0.8676 for Lasso/L1, and 0.8703 for Elastic Net.
- ROC-AUC decreased from 0.9568 for the baseline to 0.9437 for Ridge/L2, 0.9358 for Lasso/L1, and 0.9391 for Elastic Net.
- Ridge/L2 retained all 30,000 coefficients as nonzero values.
- Lasso/L1 set 29,586 of 30,000 coefficients to exactly zero (98.62%).
- Elastic Net set 27,938 of 30,000 coefficients to exactly zero (93.13%), which was fewer zero coefficients than Lasso/L1 and more than Ridge/L2.
- These observations apply to the chosen `C=1.0` and `l1_ratio=0.5` settings; no hyperparameter tuning was performed.
